# Deteksi Kakao — notebook penelitian dan website

Notebook ini menjalankan website Flask dengan **model yang sudah Anda latih**. Tekan **▶** pada sel Bagian A secara berurutan, pilih checkpoint terbaik, lalu klik **Mulai Website**. Anda tidak perlu mengetik perintah terminal.

**Sebelum mulai di VS Code:** buka folder proyek, buka notebook ini, lalu pilih **Select Kernel → Python Environments → `.venv`**. Jika kernel belum tersedia, klik dua kali **`SIAPKAN_NOTEBOOK_WINDOWS.bat`**, tunggu selesai, lalu buka kembali VS Code. Pilih interpreter yang lokasinya berada di folder proyek Anda.

- **Bagian A (4 langkah):** pemeriksaan, dependensi, pilihan checkpoint, dan kontrol website.
- **Bagian B (opsional):** audit dataset, visualisasi augmentasi, arsitektur, training baru, evaluasi, grafik, dan prediksi foto melalui notebook.
- **Run All aman:** tidak memulai website, training, evaluasi, maupun instalasi secara otomatis. Semua pekerjaan tersebut membutuhkan klik tombol.
- Tetap sertakan folder `src/`, `webapp/`, dataset, dan checkpoint. Notebook memakai fungsi bersama agar penelitian dan website konsisten.

Model: SSDLite-MobileNetV3-Large. Kelas: **Tidak Fermentasi, Setengah Fermentasi, dan Full Fermentasi**. Confidence website awal **0.20**, tanpa mengubah konfigurasi eksperimen training.


## Bagian A — membuka website tanpa terminal
### Langkah 1/4 · Temukan proyek dan periksa Python

Jalankan sel berikut. Python yang tampil sebaiknya berasal dari `.venv` proyek Anda. Sel ini tidak mengunduh apa pun dan tidak melatih model.


In [ ]:
from pathlib import Path
import gc
import html
import importlib.util
import os
import sys
from IPython.display import HTML, Image as DisplayImage, display

# Biasanya biarkan kosong. Isi hanya bila notebook dibuka di luar folder proyek.
LOKASI_PROYEK = ""
candidates = ([Path(LOKASI_PROYEK).expanduser()] if LOKASI_PROYEK else [])
candidates += [Path.cwd(), Path.cwd() / "KAKAO-MODEL", *Path.cwd().parents]
PROJECT_ROOT = next((p.resolve() for p in candidates
                     if (p / "config.py").is_file() and (p / "src").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Buka folder KAKAO-MODEL di VS Code, atau isi LOKASI_PROYEK di atas.")
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / ".cache" / "matplotlib"))
os.environ.setdefault("XDG_CACHE_HOME", str(PROJECT_ROOT / ".cache"))
print("Folder proyek:", PROJECT_ROOT)
print("Python aktif :", sys.executable)
print("Versi Python :", sys.version.split()[0])
print("Environment  :", "virtual environment" if sys.prefix != sys.base_prefix else "Python global; pilih kernel .venv")
try:
    import torch
    import torchvision
    print("PyTorch      :", torch.__version__)
    print("TorchVision  :", torchvision.__version__)
    print("CUDA tersedia:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU          :", torch.cuda.get_device_name(0))
except Exception as error:
    print("Pustaka model belum siap:", error)
    print("Pilih kernel .venv proyek yang sebelumnya digunakan untuk training. Sel dependensi web tidak memasang PyTorch.")


### Langkah 2/4 · Periksa dependensi website

Jika ada paket yang kurang, klik **Pasang Dependensi Web/Notebook** pada hasil sel. Proses hanya memasang kebutuhan website/notebook, **tanpa memasang ulang PyTorch GPU**. Tunggu selesai sebelum melanjutkan.

Jika tombol belum dapat tampil karena `ipywidgets` belum ada, gunakan `SIAPKAN_NOTEBOOK_WINDOWS.bat` melalui klik dua kali, lalu pilih ulang kernel dan jalankan notebook dari awal.


In [ ]:
import subprocess

web_modules = {
    "Flask": "flask", "Waitress": "waitress", "pillow-heif": "pillow_heif",
    "ipywidgets": "ipywidgets", "ipykernel": "ipykernel",
}

def missing_web_dependencies():
    return [name for name, module in web_modules.items() if importlib.util.find_spec(module) is None]

missing = missing_web_dependencies()
print("Paket belum tersedia:" if missing else "Seluruh dependensi web/notebook tersedia.", ", ".join(missing))
if importlib.util.find_spec("ipywidgets") is not None:
    import ipywidgets as widgets
    install_button = widgets.Button(description="Pasang Dependensi Web/Notebook", layout=widgets.Layout(width="300px"))
    install_output = widgets.Output()

    def install_web_dependencies(_):
        install_button.disabled = True
        with install_output:
            install_output.clear_output(wait=True)
            if not missing_web_dependencies():
                print("Semua paket sudah tersedia; tidak ada instalasi yang diperlukan.")
            else:
                requirement = PROJECT_ROOT / "requirements-notebook.txt"
                if not requirement.is_file():
                    print("requirements-notebook.txt belum ada. Salin seluruh pembaruan proyek, bukan notebook saja.")
                else:
                    print("Memasang paket web/notebook pada Python:", sys.executable)
                    result = subprocess.run(
                        [sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "-r", str(requirement)],
                        cwd=PROJECT_ROOT, capture_output=True, text=True,
                    )
                    print(result.stdout[-9000:])
                    if result.returncode:
                        print(result.stderr[-4000:])
                        print("Instalasi belum selesai. Periksa pesan di atas sebelum melanjutkan.")
                    else:
                        print("Selesai. Jalankan kembali sel ini, lalu lanjutkan Langkah 3 dan 4.")
        install_button.disabled = False

    install_button.on_click(install_web_dependencies)
    display(install_button, install_output)
else:
    print("Klik dua kali SIAPKAN_NOTEBOOK_WINDOWS.bat, lalu buka ulang VS Code dan pilih kernel .venv.")


### Langkah 3/4 · Pilih checkpoint dan pengaturan website

Pilih **`best_model.pth` dari eksperimen yang ingin digunakan**, misalnya folder hasil training 100 epoch. Pilihan tidak dibuat otomatis agar model lama tidak terpakai tanpa disadari. Klik **Perbarui Daftar** setelah menambah checkpoint.

Confidence `0.20` dipakai sebagai titik awal berdasarkan evaluasi validation Anda. Untuk RTX 2050, pilih perangkat **CUDA** jika tersedia. Pilihan **Otomatis** menggunakan CUDA jika tersedia, atau CPU jika tidak; perangkat yang dipakai akan ditampilkan.


In [ ]:
if importlib.util.find_spec("ipywidgets") is not None:
    import ipywidgets as widgets

    def checkpoint_options():
        paths = sorted((PROJECT_ROOT / "checkpoints").rglob("best_model.pth"))
        return [("— Pilih checkpoint terbaik —", "")] + [(str(p.relative_to(PROJECT_ROOT)), str(p)) for p in paths]

    checkpoint_widget = widgets.Dropdown(
        options=checkpoint_options(), value="", description="Checkpoint:",
        layout=widgets.Layout(width="95%"), style={"description_width": "110px"},
    )
    checkpoint_other = widgets.Text(
        value="", description="Lokasi lain:", placeholder="Opsional: path lengkap checkpoint di luar folder checkpoints",
        layout=widgets.Layout(width="95%"), style={"description_width": "110px"},
    )
    refresh_checkpoint_button = widgets.Button(description="Perbarui Daftar")
    checkpoint_list_output = widgets.Output()
    confidence_widget = widgets.BoundedFloatText(value=0.20, min=0.0, max=1.0, step=0.05, description="Confidence:")
    device_widget = widgets.Dropdown(options=[("Otomatis", "auto"), ("CUDA / NVIDIA", "cuda"), ("CPU", "cpu")], value="auto", description="Perangkat:")
    port_widget = widgets.BoundedIntText(value=5000, min=1024, max=65535, description="Port:")
    host_widget = widgets.Dropdown(options=[("Laptop ini", "127.0.0.1"), ("Jaringan lokal (lanjutan)", "0.0.0.0")], value="127.0.0.1", description="Akses:")

    def selected_checkpoint():
        choice = checkpoint_other.value.strip() or checkpoint_widget.value
        if not choice:
            raise ValueError("Pilih checkpoint pada Langkah 3 terlebih dahulu.")
        path = Path(choice).expanduser()
        if not path.is_absolute():
            path = PROJECT_ROOT / path
        if not path.is_file():
            raise FileNotFoundError(f"Checkpoint tidak ditemukan: {path}")
        return path.resolve()

    def refresh_checkpoints(_=None):
        previous = checkpoint_widget.value
        options = checkpoint_options()
        checkpoint_widget.options = options
        checkpoint_widget.value = previous if previous in [value for _, value in options] else ""
        with checkpoint_list_output:
            checkpoint_list_output.clear_output(wait=True)
            print(f"Ditemukan {len(options) - 1} checkpoint terbaik. Pilih satu secara eksplisit.")

    refresh_checkpoint_button.on_click(refresh_checkpoints)
    display(widgets.VBox([checkpoint_widget, checkpoint_other, refresh_checkpoint_button, confidence_widget,
                          device_widget, port_widget, host_widget, checkpoint_list_output]))
    refresh_checkpoints()
else:
    print("Selesaikan Langkah 2, lalu jalankan kembali sel ini.")


### Langkah 4/4 · Mulai dan hentikan website

Klik **Mulai Website**, tunggu model selesai dimuat, lalu klik **Buka Website** yang muncul. Website menyediakan Home, kamera, unggah gambar, serta riwayat maksimal 10 hasil pada browser. Riwayat tidak menggunakan database.

Server berjalan di latar belakang sehingga kernel notebook tetap dapat digunakan. Biarkan notebook dan kernel tetap aktif selama website digunakan. Klik **Hentikan Website** setelah selesai. Menjalankan sel ini ulang tidak memulai atau menghentikan server yang sudah aktif.

**Untuk ponsel:** pilih Akses jaringan lokal bila diperlukan. Ponsel dan laptop harus dapat terhubung. Kamera browser pada ponsel membutuhkan **HTTPS**, sehingga membuka alamat IP laptop melalui HTTP saja belum cukup untuk kamera. Unggahan gambar dapat dicoba lebih dahulu. Panduan HTTPS tersedia pada `README_WEB.md`.


In [ ]:
if importlib.util.find_spec("ipywidgets") is not None and "checkpoint_widget" in globals():
    # State dipertahankan bila sel kontrol dijalankan ulang.
    if "_kakao_notebook_state" not in globals():
        _kakao_notebook_state = {"manager": None, "app": None}

    def website_is_running():
        manager = _kakao_notebook_state.get("manager")
        return manager is not None and manager.is_running

    start_website_button = widgets.Button(description="Mulai Website", button_style="success", icon="play")
    stop_website_button = widgets.Button(description="Hentikan Website", button_style="warning", icon="stop", disabled=not website_is_running())
    website_output = widgets.Output()

    def show_website_link(url):
        safe_url = html.escape(url, quote=True)
        display(HTML(f'<a href="{safe_url}" target="_blank" rel="noopener" '
                     'style="display:inline-block;padding:12px 20px;background:#285943;color:white;'
                     'border-radius:10px;text-decoration:none;font-weight:600">Buka Website</a>'))
        print("Alamat lokal:", url)

    def start_website(_):
        start_website_button.disabled = True
        with website_output:
            website_output.clear_output(wait=True)
            manager = None
            try:
                if website_is_running():
                    print("Website sudah aktif. Hentikan terlebih dahulu jika ingin mengganti pengaturan.")
                    return
                if missing_web_dependencies():
                    raise RuntimeError("Selesaikan pemasangan dependensi pada Langkah 2 terlebih dahulu.")
                checkpoint = selected_checkpoint()
                from webapp.app import create_app
                from webapp.notebook_server import NotebookServerManager
                print("Memuat checkpoint:", checkpoint)
                app = create_app({"CHECKPOINT_PATH": str(checkpoint),
                                  "WEB_CONFIDENCE_THRESHOLD": float(confidence_widget.value),
                                  "DEVICE": device_widget.value})
                health_response = app.test_client().get("/api/health")
                health = health_response.get_json(silent=True) or {}
                if not health.get("ready"):
                    raise RuntimeError(health.get("error") or "Model belum siap; periksa checkpoint dan environment.")
                manager = NotebookServerManager()
                manager.start(app, host=host_widget.value, port=int(port_widget.value))
                _kakao_notebook_state.update(manager=manager, app=app)
                print("Website siap. Perangkat:", health.get("device", device_widget.value))
                print("Confidence:", confidence_widget.value)
                show_website_link(f"http://127.0.0.1:{port_widget.value}")
                if host_widget.value == "0.0.0.0":
                    print("Akses jaringan lokal aktif. Untuk kamera ponsel, ikuti panduan HTTPS di README_WEB.md.")
            except Exception as error:
                if manager is not None:
                    manager.stop()
                print("Website belum dapat dimulai:", error)
                print("Tidak ada training atau unduhan bobot yang dijalankan. Perbaiki penyebab di atas, lalu klik Mulai Website lagi.")
            finally:
                start_website_button.disabled = False
                stop_website_button.disabled = not website_is_running()

    def stop_website(_):
        with website_output:
            try:
                manager = _kakao_notebook_state.get("manager")
                if manager is not None:
                    manager.stop()
                _kakao_notebook_state.update(manager=None, app=None)
                gc.collect()
                if "torch" in globals() and torch.cuda.is_available():
                    torch.cuda.empty_cache()
                print("Website dihentikan. Anda dapat mengubah pengaturan lalu memulainya kembali.")
            except Exception as error:
                print("Periksa penghentian website:", error)
            finally:
                stop_website_button.disabled = not website_is_running()

    start_website_button.on_click(start_website)
    stop_website_button.on_click(stop_website)
    display(widgets.HBox([start_website_button, stop_website_button]), website_output)
    if website_is_running():
        print("Server dari eksekusi sel sebelumnya masih aktif. Gunakan tombol Hentikan Website untuk menutupnya.")
else:
    print("Jalankan Langkah 2 dan 3 terlebih dahulu.")


## Bagian B — penelitian dan demonstrasi opsional

Bagian ini memakai fungsi penelitian yang sudah ada di proyek. **Anda tidak harus menjalankannya untuk membuka website.** Semua proses membutuhkan tombol; Run All hanya menampilkan kontrol.

Alur: pengambilan foto → labeling bounding box di Roboflow → pemisahan train/val/test → ekspor COCO → pemeriksaan → loader/augmentasi → training → evaluasi → prediksi.

Tetapkan split sebelum membuat variasi dari foto yang sama agar variasi satu foto tidak tersebar ke train dan test. Augmentasi runtime pada kode hanya diterapkan pada train. Sel berikut tidak mengunggah data ke Roboflow.


### Tahap 2–4 · Audit dataset, loader, dan augmentasi

Letakkan ekspor COCO pada `dataset/train`, `dataset/val` (atau `valid`), serta `dataset/test`. Setiap split berisi gambar dan `_annotations.coco.json`.

Audit memeriksa anotasi, dimensi gambar, pemetaan kelas, dan duplikasi persis antar-split. Audit tidak dapat menjamin tidak ada foto sangat mirip yang sudah dipotong atau dikodekan ulang.

Tombol contoh menampilkan satu gambar train dengan anotasi asli dan contoh horizontal flip. **Untuk ilustrasi saja, contoh dipaksa flip 100%; training tetap memakai peluang 50% sesuai `config.py`.** Model SSD melakukan resize 320 × 320 dan normalisasi internal; dataset hanya mengubah gambar menjadi tensor RGB [0,1].


In [ ]:
if importlib.util.find_spec("ipywidgets") is not None:
    dataset_widget = widgets.Text(value=str(PROJECT_ROOT / "dataset"), description="Dataset:", layout=widgets.Layout(width="95%"))
    audit_button = widgets.Button(description="Periksa Dataset")
    sample_button = widgets.Button(description="Lihat Anotasi & Flip")
    dataset_output = widgets.Output()

    def audit_dataset(_):
        with dataset_output:
            dataset_output.clear_output(wait=True)
            try:
                import json
                from src.dataset import build_datasets, validate_dataset_splits
                datasets = build_datasets(Path(dataset_widget.value).expanduser())
                audit = validate_dataset_splits(datasets)
                print(json.dumps(audit, indent=2, ensure_ascii=False))
            except Exception as error:
                print("Dataset belum siap:", error)

    def show_sample(_):
        with dataset_output:
            dataset_output.clear_output(wait=True)
            try:
                import matplotlib.pyplot as plt
                from matplotlib.patches import Rectangle
                import config
                from src.dataset import load_split
                from src.transforms import RandomHorizontalFlip
                dataset = load_split(Path(dataset_widget.value).expanduser(), "train", train=False)
                image, target = dataset[0]
                flipped, flipped_target = RandomHorizontalFlip(probability=1.0)(image, target)
                fig, axes = plt.subplots(1, 2, figsize=(14, 5))
                colors = {1: "#E64646", 2: "#F5B928", 3: "#32B455"}
                for ax, (tensor, annotation, title) in zip(axes, [(image, target, "Anotasi asli"), (flipped, flipped_target, "Contoh horizontal flip")]):
                    ax.imshow(tensor.permute(1, 2, 0).numpy())
                    for box, label in zip(annotation["boxes"].tolist(), annotation["labels"].tolist()):
                        x1, y1, x2, y2 = box
                        ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor=colors[label], linewidth=1.5))
                    ax.set_title(title)
                    ax.axis("off")
                plt.tight_layout()
                plt.show()
                print("Warna anotasi: Tidak Fermentasi merah; Setengah kuning; Full hijau.")
                print("Peluang flip training sebenarnya:", config.HORIZONTAL_FLIP_PROBABILITY)
            except Exception as error:
                print("Contoh belum dapat ditampilkan:", error)

    audit_button.on_click(audit_dataset)
    sample_button.on_click(show_sample)
    display(dataset_widget, widgets.HBox([audit_button, sample_button]), dataset_output)


### Tahap 5 · Arsitektur dan transfer learning

SSDLite-MobileNetV3-Large mendeteksi banyak objek sekaligus. Head klasifikasi memakai **4 label termasuk background**, sedangkan tiga label objek dipakai untuk hitungan.

Training baru memakai bobot pretrained sesuai konfigurasi proyek. Pemulihan checkpoint memakai arsitektur tanpa mengunduh bobot pretrained lagi. Tombol berikut hanya memeriksa arsitektur pada CPU, tanpa training dan tanpa download.


In [ ]:
if importlib.util.find_spec("ipywidgets") is not None:
    model_info_button = widgets.Button(description="Lihat Ringkasan Model")
    model_info_output = widgets.Output()

    def show_model_information(_):
        with model_info_output:
            model_info_output.clear_output(wait=True)
            try:
                import config
                from src.model import build_model
                model = build_model(pretrained=False)
                print("Arsitektur:", config.MODEL_NAME)
                print("Jumlah label termasuk background:", config.NUM_CLASSES)
                print("Nama kelas:", config.CLASS_NAMES)
                print("Total parameter:", f"{sum(parameter.numel() for parameter in model.parameters()):,}")
                print("Head klasifikasi:", type(model.head.classification_head).__name__)
                print("Sumber implementasi resmi: torchvision.models.detection.ssdlite320_mobilenet_v3_large")
                del model
                gc.collect()
            except Exception as error:
                print("Ringkasan model belum tersedia:", error)

    model_info_button.on_click(show_model_information)
    display(model_info_button, model_info_output)


### Tahap 6 · Training eksperimen baru (opsional)

**Lewati tahap ini jika checkpoint sudah ada.** Hentikan website terlebih dahulu untuk menghemat VRAM. Training dapat berlangsung lama dan memakai kernel sampai selesai. Untuk berhenti darurat, gunakan **Interrupt** di toolbar notebook.

Centang persetujuan training, tentukan epoch dan batch, lalu klik tombol. Eksperimen selalu disimpan pada folder baru bertanggal sehingga tidak menimpa checkpoint lama. Batch awal 4; turunkan menjadi 2 jika GPU kehabisan memori. Ini memulai training baru, bukan resume.

Best checkpoint dipilih berdasarkan mAP@0.5:0.95 pada validation. Test belum dipakai pada tahap training. Tidak ada jumlah epoch yang menjamin semua biji terdeteksi.


In [ ]:
if importlib.util.find_spec("ipywidgets") is not None:
    train_epochs_widget = widgets.BoundedIntText(value=100, min=1, max=10000, description="Epoch:")
    train_batch_widget = widgets.BoundedIntText(value=4, min=2, max=128, description="Batch:")
    train_cuda_widget = widgets.Checkbox(value=True, description="Wajib memakai CUDA / NVIDIA", indent=False)
    train_confirm_widget = widgets.Checkbox(value=False, description="Saya ingin memulai training eksperimen baru", indent=False)
    train_button = widgets.Button(description="Mulai Training Baru", button_style="warning")
    train_output = widgets.Output()

    def train_new_experiment(_):
        with train_output:
            train_output.clear_output(wait=True)
            if not train_confirm_widget.value:
                print("Training belum dijalankan. Centang pilihan training hanya jika memang diperlukan.")
                return
            if "website_is_running" in globals() and website_is_running():
                print("Hentikan website pada Langkah 4 terlebih dahulu, lalu mulai training.")
                return
            train_button.disabled = True
            try:
                from datetime import datetime
                from uuid import uuid4
                from src.dataset import load_split
                from src.train import train_model
                from src.utils import get_device
                dataset_root = Path(dataset_widget.value).expanduser()
                train_dataset = load_split(dataset_root, "train", train=True)
                val_dataset = load_split(dataset_root, "val", train=False)
                device = get_device(require_cuda=train_cuda_widget.value)
                experiment = "notebook_" + datetime.now().strftime("%Y%m%d_%H%M%S") + "_" + uuid4().hex[:6]
                result = train_model(
                    train_dataset, val_dataset, num_epochs=int(train_epochs_widget.value),
                    batch_size=int(train_batch_widget.value), device=device,
                    checkpoint_dir=PROJECT_ROOT / "checkpoints" / experiment,
                    metrics_dir=PROJECT_ROOT / "outputs" / "metrics" / experiment,
                )
                print("Training selesai. Best checkpoint:", result["best_checkpoint"])
                print("Klik Perbarui Daftar pada Langkah 3, lalu pilih checkpoint jika ingin menggunakannya.")
                refresh_checkpoints()
            except Exception as error:
                print("Training berhenti:", error)
                print("Checkpoint yang berhasil tersimpan tetap berada pada folder eksperimen. Periksa penyebab sebelum mengulang.")
            finally:
                train_confirm_widget.value = False
                train_button.disabled = False
                gc.collect()
                if "torch" in globals() and torch.cuda.is_available():
                    torch.cuda.empty_cache()

    train_button.on_click(train_new_experiment)
    display(widgets.VBox([train_epochs_widget, train_batch_widget, train_cuda_widget, train_confirm_widget, train_button, train_output]))


### Tahap 7 · Evaluasi checkpoint dan tampilkan grafik

Pilih checkpoint pada Langkah 3 dan hentikan website sebelum evaluasi agar model tidak memakai VRAM bersamaan. Pilihan confidence menggunakan kontrol Langkah 3.

- **Validation:** untuk membandingkan threshold atau memilih pengaturan.
- **Test:** gunakan setelah pilihan model dan pengaturan ditetapkan; jangan dipakai berulang untuk memilih threshold.
- mAP tetap dihitung dengan ambang skor rendah sesuai protokol proyek. Confidence operasional memengaruhi precision, recall, F1, dan hitungan.

Tombol grafik hanya membaca CSV yang sudah tersimpan, tanpa menjalankan training.


In [ ]:
if importlib.util.find_spec("ipywidgets") is not None:
    evaluation_split_widget = widgets.Dropdown(options=[("Validation", "val"), ("Test akhir", "test")], value="val", description="Split:")
    test_confirm_widget = widgets.Checkbox(value=False, description="Model/pengaturan sudah final; saya ingin evaluasi test", indent=False)
    evaluate_button = widgets.Button(description="Evaluasi Checkpoint")
    evaluation_output = widgets.Output()

    def notebook_device():
        import torch
        choice = device_widget.value
        if choice == "cuda" and not torch.cuda.is_available():
            raise RuntimeError("CUDA diminta tetapi tidak tersedia. Periksa kernel .venv dan instalasi GPU.")
        return torch.device("cuda" if choice == "cuda" or (choice == "auto" and torch.cuda.is_available()) else "cpu")

    def load_notebook_model():
        import config
        import torch
        from src.model import build_model
        from src.utils import load_checkpoint
        device = notebook_device()
        if device.type == "cpu":
            torch.set_num_threads(config.CPU_NUM_THREADS)
        checkpoint = load_checkpoint(selected_checkpoint(), device)
        model = build_model(pretrained=False).to(device)
        model.load_state_dict(checkpoint["model_state_dict"], strict=True)
        model.eval()
        return model, device

    def evaluate_checkpoint(_):
        with evaluation_output:
            evaluation_output.clear_output(wait=True)
            if "website_is_running" in globals() and website_is_running():
                print("Hentikan website terlebih dahulu untuk menghemat memori GPU.")
                return
            if evaluation_split_widget.value == "test" and not test_confirm_widget.value:
                print("Konfirmasikan evaluasi test hanya setelah model dan pengaturan ditetapkan.")
                return
            model = None
            evaluate_button.disabled = True
            try:
                import json
                from datetime import datetime
                from uuid import uuid4
                from torch.utils.data import DataLoader
                from src.dataset import load_split
                from src.evaluate import evaluate_model
                from src.utils import collate_fn, save_metrics_report
                model, device = load_notebook_model()
                split = evaluation_split_widget.value
                dataset = load_split(Path(dataset_widget.value).expanduser(), split, train=False)
                loader = DataLoader(dataset, batch_size=2, shuffle=False, num_workers=0, pin_memory=False, collate_fn=collate_fn)
                metrics = evaluate_model(model, loader, device, confidence_threshold=float(confidence_widget.value))
                directory = PROJECT_ROOT / "outputs" / "metrics" / ("evaluation_" + datetime.now().strftime("%Y%m%d_%H%M%S") + "_" + uuid4().hex[:6])
                save_metrics_report(directory, split, metrics)
                print(json.dumps({key: metrics[key] for key in ("map", "map_50", "precision", "recall", "f1", "tp", "fp", "fn")}, indent=2))
                print("Metrik lengkap dan laporan per kelas tersimpan di:", directory)
            except Exception as error:
                print("Evaluasi belum selesai:", error)
            finally:
                del model
                gc.collect()
                if "torch" in globals() and torch.cuda.is_available():
                    torch.cuda.empty_cache()
                evaluate_button.disabled = False

    history_options = [("— Pilih training_history.csv —", "")] + [(str(p.relative_to(PROJECT_ROOT)), str(p)) for p in sorted((PROJECT_ROOT / "outputs").rglob("training_history.csv"))]
    history_widget = widgets.Dropdown(options=history_options, value="", description="History:", layout=widgets.Layout(width="95%"))
    plot_button = widgets.Button(description="Tampilkan Grafik")
    refresh_history_button = widgets.Button(description="Perbarui Daftar Grafik")
    plot_output = widgets.Output()

    def refresh_history(_):
        current = history_widget.value
        options = [("— Pilih training_history.csv —", "")] + [(str(p.relative_to(PROJECT_ROOT)), str(p)) for p in sorted((PROJECT_ROOT / "outputs").rglob("training_history.csv"))]
        history_widget.options = options
        history_widget.value = current if current in [value for _, value in options] else ""

    def plot_history(_):
        with plot_output:
            plot_output.clear_output(wait=True)
            if not history_widget.value:
                print("Pilih CSV training terlebih dahulu.")
                return
            try:
                import csv
                import matplotlib.pyplot as plt
                with Path(history_widget.value).open(encoding="utf-8-sig", newline="") as handle:
                    rows = list(csv.DictReader(handle))
                if not rows:
                    raise ValueError("CSV tidak memiliki baris epoch.")
                epochs = [int(row["epoch"]) for row in rows]
                fig, axes = plt.subplots(1, 2, figsize=(13, 4))
                for key, label in [("training_loss", "Total loss"), ("classification", "Classification"), ("bbox_regression", "BBox")]:
                    if key in rows[0]:
                        axes[0].plot(epochs, [float(row[key]) for row in rows], label=label)
                for key, label in [("val_map", "mAP@0.5:0.95"), ("val_map_50", "mAP@0.5"), ("val_precision", "Precision"), ("val_recall", "Recall"), ("val_f1", "F1")]:
                    if key in rows[0]:
                        axes[1].plot(epochs, [float(row[key]) for row in rows], label=label)
                for ax, title in zip(axes, ["Loss training", "Metrik validation"]):
                    ax.set_title(title)
                    ax.set_xlabel("Epoch")
                    ax.grid(alpha=0.2)
                    ax.legend()
                plt.tight_layout()
                plt.show()
            except Exception as error:
                print("Grafik belum dapat ditampilkan:", error)

    evaluate_button.on_click(evaluate_checkpoint)
    plot_button.on_click(plot_history)
    refresh_history_button.on_click(refresh_history)
    display(widgets.VBox([evaluation_split_widget, test_confirm_widget, evaluate_button, evaluation_output,
                          history_widget, widgets.HBox([refresh_history_button, plot_button]), plot_output]))


### Tahap 8 · Prediksi foto baru di notebook (opsional)

Selain melalui website, Anda dapat memilih satu foto dari tombol di bawah. Pilih checkpoint pada Langkah 3 terlebih dahulu. Hasil memakai confidence pada Langkah 3 dan disimpan pada folder unik di `outputs/predictions/notebook_*`.

Hentikan website sebelum tombol prediksi notebook dipakai. Foto baru di luar dataset tidak perlu dilabeli untuk inferensi. Jumlah adalah **biji yang berhasil dideteksi**, sehingga tetap perlu dibandingkan dengan kondisi foto sebenarnya.


In [ ]:
if importlib.util.find_spec("ipywidgets") is not None:
    photo_widget = widgets.FileUpload(accept=".jpg,.jpeg,.png,.webp,.heic,.heif", multiple=False, description="Pilih Foto Baru")
    predict_button = widgets.Button(description="Deteksi Foto", button_style="success")
    prediction_output = widgets.Output()

    def predict_uploaded_photo(_):
        with prediction_output:
            prediction_output.clear_output(wait=True)
            if "website_is_running" in globals() and website_is_running():
                print("Hentikan website terlebih dahulu, atau lakukan deteksi langsung melalui halaman Unggah Gambar website.")
                return
            if not photo_widget.value:
                print("Pilih foto terlebih dahulu.")
                return
            model = None
            predict_button.disabled = True
            try:
                import io
                from uuid import uuid4
                from PIL import Image, ImageOps
                if importlib.util.find_spec("pillow_heif") is not None:
                    from pillow_heif import register_heif_opener
                    register_heif_opener()
                upload = photo_widget.value[0]
                content = bytes(upload["content"])
                if len(content) > 15 * 1024 * 1024:
                    raise ValueError("Ukuran foto maksimal 15 MB.")
                destination = PROJECT_ROOT / "outputs" / "predictions" / ("notebook_" + uuid4().hex[:12])
                destination.mkdir(parents=True, exist_ok=False)
                source = destination / "foto_input.png"
                with Image.open(io.BytesIO(content)) as image:
                    ImageOps.exif_transpose(image).convert("RGB").save(source)
                from src.predict import predict_image
                model, device = load_notebook_model()
                result = predict_image(model, source, device, confidence_threshold=float(confidence_widget.value), output_dir=destination)
                display(DisplayImage(filename=result["annotated_image"], width=900))
                for name, count in result["counts"].items():
                    print(f"{name}: {count}")
                print("Total biji terdeteksi:", result["total"])
                print("Folder gambar dan JSON:", destination)
            except Exception as error:
                print("Prediksi belum selesai:", error)
            finally:
                del model
                gc.collect()
                if "torch" in globals() and torch.cuda.is_available():
                    torch.cuda.empty_cache()
                predict_button.disabled = False

    predict_button.on_click(predict_uploaded_photo)
    display(widgets.VBox([photo_widget, predict_button, prediction_output]))


## Selesai dan catatan sidang

Untuk demonstrasi, gunakan checkpoint yang sudah selesai dilatih: **Bagian A → pilih checkpoint → Mulai Website → unggah foto baru**. Training ulang bukan syarat demo.

Riwayat website disimpan di browser/perangkat yang sama melalui localStorage; data bisa hilang bila data browser dihapus. Hasil resolusi penuh dapat diunduh dari hasil deteksi yang masih aktif. Riwayat hanya menyimpan thumbnail terkompresi dan hitungan.

Setelah selesai, klik **Hentikan Website** dan hentikan kamera pada halaman browser. Restart/shutdown kernel juga menghentikan server notebook. Jika memakai komputer lain, lingkungan Python dan paket perlu disiapkan kembali; folder `.venv` bukan berkas model yang dipindahkan antar-komputer.

Panduan: `README_NOTEBOOK.md` untuk notebook, `README_WEB.md` untuk website, dan `README.md` untuk kode penelitian.
